# Kaggle コミットで任意の config を1本学習するテンプレ

リポジトリ内の **好きな YAML config を指定して1本学習する**ノートです。Step 3（タイム短縮）の個人実験や、Step 4b（Hardcore のタイム短縮）など、「sweep ではなく特定の設定を回したい」とき全般に使えます。準備手順（clone → pip install → W&B ログイン）は sweep 用ノートと同じで、最後のセルだけが違います。

**注意事項（読んでから実行）**

- このノートは **CPU セッション** で使う。GPU は選択しない。
- **インターネット接続を有効化** すること（Settings → Internet を On。電話番号認証が必要）。
- W&B の API キーは Add-ons → Secrets に `WANDB_API_KEY` として登録し、チェックボックスを On にする。
- **編集するのは最後のセルの変数 2 つだけ**（`CONFIG_PATH` / `RESUME_RUN_PATH`）。
- 実測で 100万ステップ ≒ 8〜10 時間。セッション上限（約12時間）には1本しか収まらない。
- 学習が終わると、ベストモデルの走りの動画が **自動で W&B の run にアップロードされる**（train.py の機能）。
- 使い方: 右上の Save Version → 「**Save & Run All（Commit）**」で放置実行する。


In [ ]:
# 1. リポジトリを clone
# 【なぜ】Kaggle が用意するマシンはまっさらで、私たちのコードが入っていない。
#         まず GitHub からコード一式をこのマシンにダウンロードする。
!git clone https://github.com/shironaganegi/SB3-Team.git
%cd SB3-Team

In [ ]:
# 2. 依存ライブラリをインストール
# 【なぜ】学習コードは PyTorch・gymnasium・sb3-contrib などに依存しているが、
#         まっさらなマシンには入っていない（または版が合わない）ため。
!pip install -r requirements.txt

In [ ]:
# 3. W&B にログイン
# 【なぜ】学習結果と動画を W&B のダッシュボードに送るため。
#         キーは Add-ons → Secrets の WANDB_API_KEY から読む（直書き禁止）。
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
!wandb login --relogin $WANDB_API_KEY

In [ ]:
# 4. 指定した config で1本学習（このノートの本体）
# 【なぜ】sweep は「司令塔が配る設定」を回すものだが、Step 3 以降は
#         「自分が決めた特定の設定」を回したい場面が増える。このセルは
#         リポジトリ内の任意の YAML を指定してそのまま学習する。
# 【成功すると】W&B に run が1本増え、学習後にモデルと動画が自動で上がる。
#
# 【書き換える場所】下の2つの変数だけ。
#   CONFIG_PATH     … 使う config のパス（リポジトリルートからの相対パス）。
#                     個人実験は members/<自分の番号>/configs/ に置いたものを指定。
#   RESUME_RUN_PATH … 空文字 "" なら新規学習。W&B の run パス
#                     （例: sai3desuyo-/bipedal-timetrial/xsrplaip）を入れると、
#                     その run の model.zip をダウンロードして追加学習する。
#                     ※再開時は TQC ハイパラ（lr など）はチェックポイント側の値が
#                       使われ、YAML の値は反映されない（env_id・vel_coef・
#                       total_timesteps などの学習制御キーは反映される）。

CONFIG_PATH = "members/0375/configs/velcoef2_lr4.4e-4_seed1.yaml"
RESUME_RUN_PATH = ""

import wandb, yaml

# --- （指定があれば）ベースモデルをダウンロード ---
resume_arg = ""
if RESUME_RUN_PATH:
    api = wandb.Api()
    base_run = api.run(RESUME_RUN_PATH)
    model_file = [f for f in base_run.files() if f.name == "model.zip"]
    assert model_file, f"{RESUME_RUN_PATH} に model.zip がありません（Finished した run を指定してください）"
    model_file[0].download(root="base_model", replace=True)
    resume_arg = "--resume base_model/model.zip"
    print(f"[ok] base_model/model.zip を取得（{base_run.name}: eval_reward={base_run.summary.get('eval/mean_reward')}）")

# --- config のコピーを作る（W&B を強制的に有効化）---
cfg = yaml.safe_load(open(CONFIG_PATH))
cfg["use_wandb"] = True
yaml.safe_dump(cfg, open("run_config.yaml", "w"), allow_unicode=True)
print(f"[ok] {CONFIG_PATH} をベースに run_config.yaml を作成")

# --- 学習を実行 ---
!python src/train.py --config run_config.yaml {resume_arg}